In [0]:
import data.utilities.common as Commonlib
import yaml
from beertech_utils.data.integrity.integrity_check import EmptyDataFrameCheck
from data.utilities.extractors import MSSQLExtractor
from data.utilities.medallion import Medallion
from data.utilities.medallion_layer import MedallionLayer as ML
from pyspark.sql.functions import col


In [0]:
# this entire command block will re-execute each time the "1. Subject Area" drop-down selection is changed - this populates the appropriate configurations in the "2. Configuration File" drop-down
CONFIG_BASE_PATH = "../../configuration/mssql/"

subject_entries = Commonlib.list_directories(CONFIG_BASE_PATH)
dbutils.widgets.dropdown("subject_area", "", [""] + sorted(subject_entries), "1. Subject Area")
subject_area = dbutils.widgets.get("subject_area")

config_selection_path = f"{CONFIG_BASE_PATH}{subject_area}"
config_entries = Commonlib.list_files(config_selection_path, max_depth=1)
dbutils.widgets.dropdown("config_file", "", [""] + sorted(config_entries), "2. Configuration File")

In [0]:
# load configs using parameter values
config_file = dbutils.widgets.get("config_file")
config_file_path = f"{CONFIG_BASE_PATH}{subject_area}/{config_file}"

config = Commonlib.get_object_config(config_file_path)
source_config = config.get("source")
bronze_config = config.get(ML.bronze)

In [0]:
try:
    medallion = Medallion(config)

    mssql_db = source_config.get("database")
    mssql_schema = source_config.get("schema")
    mssql_table = source_config.get("name", config.get("name", ""))
    mssql_query = source_config.get("query", "")
    mssql_fqtn = f"{mssql_db}.{mssql_schema}.{mssql_table}"

    mssql_extract = mssqlExtractor(secret_key=source_config.get("connection"))

    if mssql_query:
        medallion.df = mssql_extract.query(mssql_query)
    else:
        medallion.df = mssql_extract.load(mssql_fqtn)

    # destroy mssql extractor object
    del mssql_extract
    # cache the result of the first evaluation
    medallion.df.cache()

    watermark_col = bronze_config.get("watermark", "")
    load_type = bronze_config.get("load_type")

    # incremental load - based on watermark
    if load_type == "append" and watermark_col:
        watermark = medallion.get_watermark(layer=ML.bronze, watermark_col=watermark_col)
        medallion.df = medallion.df.filter(col(watermark_col) > watermark)
        medallion.logger.info(
            f"Performing incremental data load from mssql based on records where {watermark_col} > {watermark}"
        )
    # full load
    elif load_type == "overwrite":
        empty_df = EmptyDataFrameCheck(dataset_name=medallion.name)
        empty_df.run(medallion.df)
        medallion.logger.info("Performing full data load from mssql")
    # custom query load
    else:
        medallion.logger.info("Appending records from mssql based on custom query")
except Exception as e:
    medallion.logger.error(
        f"An error occured while processing the mssql_procedure job. subject={subject_area}, config_file={config_file}: {e}"
    )
    raise e

In [0]:
try:
    medallion.write_to_delta(
        load_type=load_type,
        layer=ML.bronze,
        source=f"MSSQLExtract | {mssql_fqtn}",
    )

    medallion.df.unpersist()
except Exception as e:
    medallion.logger.error(
        f"An error occured while writing to bronze via mssql_procedure. subject={subject_area}, config_file={config_file}: {e}"
    )
    raise e